In [ ]:
# ??????? ???? ? ??? ???????
from sklearn.datasets import fetch_openml
import pandas as pd
dataset = fetch_openml(data_id = 560 , as_frame =True)
df = dataset.frame

df = df.rename(columns={'class':'target'})


Q1 = df.quantile(0.25)       # ??? ???? ??? ???
Q3 = df.quantile(0.75)
IQR = Q3 - Q1
outliers = (df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))
df_clean = df[~outliers.any(axis=1)]
df_clean.hist(bins=20, figsize=(18, 14), edgecolor='black')

# ???? ????? ???? ? ????? ???? ?? ? ???
df_clean['BMI'] = (df_clean['Weight'] * 703) / (df_clean['Height'] ** 2)
df_clean.drop(['Weight', 'Height'], axis=1, inplace=True)

In [ ]:
# ????? ???? ?? ?? ??? ? ????

from sklearn.model_selection import train_test_split

x = df_clean.drop(columns='target')     # ???????? (DataFrame)
y = df_clean['target']                  # ????? ??? (Series)

x_train, x_test, y_train, y_test = train_test_split(x , y , test_size=0.2 , random_state=42)

In [ ]:
# ????? ???? ????
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import AdaBoostRegressor

ada_pipeline = Pipeline([
    ('scaler',StandardScaler()),
    ('model',AdaBoostRegressor(
        n_estimators=100,
        learning_rate= 0.8,
        random_state = 42
    ))
])

In [ ]:
# ??? ???? ???
ada_pipeline.fit(x_train,y_train)

In [ ]:
# ??? ???? ???
y_pred = ada_pipeline.predict(x_test)

In [ ]:
# ??????? ?????? ???
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
MAE = mean_absolute_error(y_test,y_pred)
MSE = mean_squared_error(y_test,y_pred)
R2 = r2_score(y_test,y_pred)
MAE ,MSE , R2

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators':[50,100,150],
    'model__learning_rate':[1.0,0.5,0.1,0.01]
}

grid_search = GridSearchCV(
    estimator = ada_pipeline,
    param_grid=param_grid,
    cv = 5,
    scoring = 'r2',
    n_jobs = -1
)


In [ ]:
grid_search.fit(x_train, y_train)

In [ ]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(x_test)

# ??????? ?????? ???
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
MAE = mean_absolute_error(y_test,y_pred_best)
MSE = mean_squared_error(y_test,y_pred_best)
R2 = r2_score(y_test,y_pred_best)
MAE ,MSE , R2